In [1]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import os
print(os.getcwd())

import findspark
findspark.init()

from pyspark.sql import functions as F

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col, lit, trim, length, count, countDistinct, coalesce, desc, when,
    try_to_timestamp, year,
    sum as spark_sum,
    min as spark_min,
    max as spark_max,
    round as spark_round
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType
)

/home/ubuntu/spark-notebooks/NYC311-Pluto/notebooks


spark = (
    SparkSession.builder \
    .appName("DAT535-Project-NYC311-silver2") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "5g") \
    .getOrCreate()
)
print("Spark is ready!")
print("Spark version:", spark.version)
spark.sparkContext.setLogLevel("WARN")


In [3]:
spark = (
    SparkSession.builder
    .appName("DAT535-Project-NYC311-silver2")
    .master("spark://192.168.18.168:7077")
    .config("spark.driver.host", "192.168.18.168")
    .config("spark.cores.max", "8")
    .config("spark.executor.cores", "4")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Connected Master:", spark.sparkContext.master)

TypeError: 'JavaPackage' object is not callable

# 1. load dataframe

In [ ]:
# Path to the original NYC 311 dataset
nyc311_path = "/home/ubuntu/spark-project-data/bronze/nyc311/nyc311_2020_present.csv"

# Reload the raw dataset
nyc311_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(nyc311_path)
)

print("NYC 311 raw DataFrame restored successfully!")
print("Number of columns:", len(nyc311_raw_df.columns))

# 2. Read the NYC-311 raw CSV

In [ ]:
nyc311_raw_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv("/home/ubuntu/spark-project-data/bronze/nyc311/nyc311_2020_present.csv")

print("NYC 311 dataset loaded!")
print("Number of columns:", len(nyc311_raw_df.columns))
print(nyc311_raw_df.columns)

# 3. Inspect Unique Key and BBL

In [ ]:
nyc311_raw_df.select(
    "Unique Key",
    "BBL"
).show(20, truncate=False)

# complaint about noise on a street may not have a specific property BBL.

# 4. Unique Key → request_id

In [ ]:
nyc311_fields_df = nyc311_raw_df.withColumn(
    "request_id",
    F.trim(F.col("Unique Key"))
)

nyc311_fields_df.select(
    "Unique Key",
    "request_id"
).show(10, truncate=False)

# 5. Validate request ID formats

In [ ]:
# A valid request ID should contain only digits.

nyc311_fields_df.groupBy(
    F.when(
        F.col("request_id").isNull() |
        (F.col("request_id") == ""),
        "Missing"
    ).when(
        F.col("request_id").rlike(r"^[0-9]+$"),
        "Valid"
    ).otherwise("Invalid").alias("request_id_status")
).count().show()

# 6. Create a cleaned BBL column

In [ ]:
nyc311_fields_df = (
    nyc311_fields_df
    .withColumn("bbl_raw", F.col("BBL"))
    .withColumn(
        "bbl",
        F.when(
            F.trim(F.col("BBL")).rlike(r"^[1-5][0-9]{9}$"),
            F.trim(F.col("BBL"))
        ).otherwise(F.lit(None).cast("string"))
    )
)

nyc311_fields_df.select(
    "BBL",
    "bbl_raw",
    "bbl"
).show(20, truncate=False)

# 7. Create bbl_status and count each category


In [ ]:
nyc311_fields_df = nyc311_fields_df.withColumn(
    "bbl_status",
    F.when(
        F.col("bbl_raw").isNull() |
        (F.trim(F.col("bbl_raw")) == ""),
        "missing"
    ).when(
        F.col("bbl").isNotNull(),
        "valid"
    ).otherwise("invalid")
)

nyc311_fields_df.groupBy("bbl_status").count().show()

# 8. Clean the Borough column

In [ ]:
# Where is service demand concentrated?
# "Brooklyn " --> "BROOKLYN"

nyc311_fields_df = (
    nyc311_fields_df
    .withColumn("borough_raw", F.col("Borough"))
    .withColumn(
        "borough",
        F.when(
            F.col("Borough").isNull() |
            (F.trim(F.col("Borough")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("Borough")))
        )
    )
)

nyc311_fields_df.groupBy("borough") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(15, truncate=False)

# 9. Create borough_status

In [ ]:
# Where is service demand concentrated?

nyc311_fields_df = nyc311_fields_df.withColumn(
    "borough_status",
    F.when(
        F.col("borough").isNull(),
        "missing"
    ).when(
        F.col("borough") == "UNSPECIFIED",
        "unspecified"
    ).when(
        F.col("borough").isin(
            "BROOKLYN",
            "QUEENS",
            "BRONX",
            "MANHATTAN",
            "STATEN ISLAND"
        ),
        "valid"
    ).otherwise("unexpected")
)

nyc311_fields_df.groupBy("borough_status") \
    .count() \
    .show()

# 10. Explore the Community Board column

In [ ]:
# New York City is divided into community districts, each represented by a Community Board.
# allows to analyze service demand at a smaller geographic level than the borough
# example:
# 01 BROOKLYN — Brooklyn Community District 1
# 02 MANHATTAN — Manhattan Community District 2

nyc311_fields_df.groupBy("Community Board") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

# output: 12 BRONX = Community Board 12 in the Bronx.S

# 11. Check Community Board value formats

In [ ]:
nyc311_fields_df.groupBy(
    F.when(
        F.col("Community Board").isNull() |
        (F.trim(F.col("Community Board")) == ""),
        "missing"
    ).when(
        F.upper(F.trim(F.col("Community Board"))).contains("UNSPECIFIED"),
        "unspecified"
    ).when(
        F.upper(F.trim(F.col("Community Board"))).rlike(
            r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
        ),
        "numbered_format"
    ).otherwise("unexpected").alias("community_board_format")
).count().show()

# 12. Investigate the 4 unexpected Community Board values

In [ ]:
nyc311_fields_df.filter(
    F.col("Community Board").isNotNull() &
    (F.trim(F.col("Community Board")) != "") &
    (~F.upper(F.trim(F.col("Community Board"))).contains("UNSPECIFIED")) &
    (~F.upper(F.trim(F.col("Community Board"))).rlike(
        r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
    ))
).select(
    "Unique Key",
    "Borough",
    "Community Board"
).show(10, truncate=False)

# 13. Inspect other columns in these 4 records

In [ ]:
nyc311_fields_df.filter(
    F.col("request_id").isin(
        "69488658",
        "65647828",
        "48556313",
        "48187012"
    )
).select(
    "Unique Key",
    "Created Date",
    "Agency",
    "Problem (formerly Complaint Type)",
    "Status",
    "Borough",
    "Community Board",
    "BBL"
).show(10, truncate=False)

# 14. Clean Community Board and create its quality status

In [ ]:
nyc311_fields_df = (
    nyc311_fields_df

    # Preserve the original Community Board
    .withColumn(
        "community_board_raw",
        F.col("Community Board")
    )

    # Create the cleaned Community Board
    .withColumn(
        "community_board",
        F.when(
            F.upper(F.trim(F.col("Community Board"))).rlike(
                r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
            ),
            F.upper(F.trim(F.col("Community Board")))
        ).otherwise(F.lit(None).cast("string"))
    )

    # Classify the Community Board quality
    .withColumn(
        "community_board_status",
        F.when(
            F.col("Community Board").isNull() |
            (F.trim(F.col("Community Board")) == ""),
            "missing"
        ).when(
            F.upper(F.trim(F.col("Community Board"))).contains("UNSPECIFIED"),
            "unspecified"
        ).when(
            F.col("community_board").isNotNull(),
            "numbered_format"
        ).otherwise("unexpected")
    )
)

nyc311_fields_df.groupBy("community_board_status").count().show()

# 15. Validate Community Board numbers

In [ ]:
# For example, 12 BRONX is valid, but 99 BRONX has the correct text format without representing a standard community district.

# Extract the board number and borough
board_number = F.substring("community_board", 1, 2).cast("int")

board_borough = F.regexp_extract(
    "community_board",
    r"^[0-9]{2} (.+)$",
    1
)

# Define the maximum valid board number per borough
max_board = (
    F.when(board_borough == "BRONX", 12)
     .when(board_borough == "BROOKLYN", 18)
     .when(board_borough == "MANHATTAN", 12)
     .when(board_borough == "QUEENS", 14)
     .when(board_borough == "STATEN ISLAND", 3)
     .otherwise(0)
)

# Find numbered-format boards outside their valid range
nyc311_fields_df.filter(
    (F.col("community_board_status") == "numbered_format") &
    (
        (board_number < 1) |
        (board_number > max_board)
    )
).groupBy("community_board") \
 .count() \
 .orderBy(F.desc("count")) \
 .show(25, truncate=False)

# 16. Update community_board_status

In [ ]:
special_areas = [
    "64 MANHATTAN",
    "55 BROOKLYN",
    "56 BROOKLYN",
    "26 BRONX",
    "27 BRONX",
    "28 BRONX",
    "80 QUEENS",
    "81 QUEENS",
    "82 QUEENS",
    "83 QUEENS",
    "84 QUEENS",
    "95 STATEN ISLAND"
]

nyc311_fields_df = nyc311_fields_df.withColumn(
    "community_board_status",
    F.when(
        F.col("community_board").isin(special_areas),
        "special_area"
    ).otherwise(F.col("community_board_status"))
)

nyc311_fields_df.groupBy("community_board_status").count().show()

# 17. Explore the Incident Zip column

In [ ]:
# Where is service demand concentrated?
# It represents the ZIP code associated with the location of a NYC 311 complaint.
# ZIP codes should remain strings, not integers

nyc311_fields_df.groupBy("Incident Zip") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

# 18. Check Incident ZIP formats

In [ ]:
nyc311_fields_df.groupBy(
    F.when(
        F.col("Incident Zip").isNull() |
        (F.trim(F.col("Incident Zip")) == ""),
        "missing"
    ).when(
        F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}$"),
        "five_digit"
    ).when(
        F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}-[0-9]{4}$"),
        "zip_plus_four"
    ).otherwise("unexpected").alias("zip_format")
).count().show()

# 19. Investigate the 25 unexpected ZIP codes

In [ ]:
nyc311_fields_df.filter(
    F.col("Incident Zip").isNotNull() &
    (F.trim(F.col("Incident Zip")) != "") &
    (~F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}$")) &
    (~F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}-[0-9]{4}$"))
).groupBy("Incident Zip") \
 .count() \
 .orderBy(F.desc("count")) \
 .show(30, truncate=False)

# 20. Clean Incident Zip and create its quality status

In [ ]:
nyc311_fields_df = (
    nyc311_fields_df

    # Step 1: Preserve the original ZIP code
    .withColumn(
        "incident_zip_raw",
        F.col("Incident Zip")
    )

    # Step 2: Create the cleaned ZIP code
    .withColumn(
        "incident_zip",
        F.when(
            F.trim(F.col("Incident Zip")).rlike(r"^[0-9]{5}$"),
            F.trim(F.col("Incident Zip"))
        ).otherwise(F.lit(None).cast("string"))
    )

    # Step 3: Create a ZIP quality status
    .withColumn(
        "incident_zip_status",
        F.when(
            F.col("Incident Zip").isNull() |
            (F.trim(F.col("Incident Zip")) == ""),
            "missing"
        ).when(
            F.col("incident_zip").isNotNull(),
            "valid_format"
        ).otherwise("invalid_format")
    )
)

# Step 4: Count records by ZIP quality status
nyc311_fields_df.groupBy("incident_zip_status").count().show()

# 21. Explore complaint types

In [ ]:
#Which types of service complaints are most common?
#Which properties experience repeated service problems?
#Which property types experience a wide variety of city-service needs?

nyc311_fields_df.groupBy(
    "Problem (formerly Complaint Type)"
).count() \
 .orderBy(F.desc("count")) \
 .show(20, truncate=False)

# 22. Clean the complaint_type column

In [ ]:
# create two new columns complaint_type_raw, complaint_type
# Identify properties with repeated service problems.
# Identify property types associated with a wide variety of city-service needs.

nyc311_fields_df = (
    nyc311_fields_df

    # Preserve the original complaint type
    .withColumn(
        "complaint_type_raw",
        F.col("Problem (formerly Complaint Type)")
    )

    # Create cleaned complaint type
    .withColumn(
        "complaint_type",
        F.when(
            F.col("complaint_type_raw").isNull() |
            (F.trim(F.col("complaint_type_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("complaint_type_raw"))
        )
    )
)

nyc311_fields_df.select(
    "complaint_type_raw",
    "complaint_type"
).show(20, truncate=False)

# 23. Check missing and modified complaint types

In [ ]:
nyc311_fields_df.select(
    F.sum(
        F.when(
            F.col("complaint_type").isNull(), 1
        ).otherwise(0)
    ).alias("missing_complaint_types"),

    F.sum(
        F.when(
            F.col("complaint_type_raw") != F.col("complaint_type"),
            1
        ).otherwise(0)
    ).alias("trimmed_complaint_types")
).show()

# 24. Explore Problem Detail (formerly Descriptor)

In [ ]:
# complaint_detail provides more specific information than complaint_type
# Identify properties with repeated service problems.
# Which properties have repeated service problems?

nyc311_fields_df.groupBy(
    "Problem Detail (formerly Descriptor)"
).count() \
 .orderBy(F.desc("count")) \
 .show(20, truncate=False)

# 25. Check missing and placeholder problem details

In [ ]:
nyc311_fields_df.select(
    F.sum(
        F.when(
            F.col("Problem Detail (formerly Descriptor)").isNull(),
            1
        ).otherwise(0)
    ).alias("null_details"),

    F.sum(
        F.when(
            F.trim(F.col("Problem Detail (formerly Descriptor)")) == "",
            1
        ).otherwise(0)
    ).alias("empty_details"),

    F.sum(
        F.when(
            F.upper(F.trim(
                F.col("Problem Detail (formerly Descriptor)")
            )) == "N/A",
            1
        ).otherwise(0)
    ).alias("na_details")
).show()

# 26. Clean the complaint_detail column

In [ ]:
nyc311_fields_df = (
    nyc311_fields_df

    # Step 1: Preserve the original problem detail
    .withColumn(
        "complaint_detail_raw",
        F.col("Problem Detail (formerly Descriptor)")
    )

    # Step 2: Create the cleaned problem detail
    .withColumn(
        "complaint_detail",
        F.when(
            F.col("complaint_detail_raw").isNull() |
            (F.trim(F.col("complaint_detail_raw")) == "") |
            (F.upper(F.trim(F.col("complaint_detail_raw"))) == "N/A"),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("complaint_detail_raw"))
        )
    )
)

nyc311_fields_df.select(
    "complaint_detail_raw",
    "complaint_detail"
).show(20, truncate=False)

# 27. Verify the cleaned complaint_detail column

In [ ]:
nyc311_fields_df.select(
    F.sum(
        F.when(
            F.col("complaint_detail").isNull(), 1
        ).otherwise(0)
    ).alias("cleaned_null_details"),

    F.sum(
        F.when(
            F.upper(F.trim(F.col("complaint_detail"))) == "N/A", 1
        ).otherwise(0)
    ).alias("remaining_na_details")
).show()

# 28. Explore the Agency column

In [ ]:
# An agency is a city department responsible for handling certain complaints.
# NYPD handles parking and noise complaints, while HPD handles housing-related complaints.
# Analyze service demand and resolution performance by agency.
# Where is resolution performance slow or inconsistent?,

nyc311_fields_df.groupBy(
    "Agency"
).count() \
 .orderBy(F.desc("count")) \
 .show(30, truncate=False)

# 29. Check missing values and extra spaces in Agency

In [ ]:
nyc311_fields_df.select(

    # Count missing or empty agency values
    F.sum(
        F.when(
            F.col("Agency").isNull() |
            (F.trim(F.col("Agency")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_agency"),

    # Count agency values containing extra surrounding spaces
    F.sum(
        F.when(
            F.col("Agency") != F.trim(F.col("Agency")),
            1
        ).otherwise(0)
    ).alias("agency_with_extra_spaces")

).show()

# 30. Create the cleaned agency column

In [ ]:
# create two columns - agency_raw, agency

nyc311_fields_df = (
    nyc311_fields_df

    # Step 1: Preserve original agency
    .withColumn(
        "agency_raw",
        F.col("Agency")
    )

    # Step 2: Create cleaned agency
    .withColumn(
        "agency",
        F.when(
            F.col("agency_raw").isNull() |
            (F.trim(F.col("agency_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("agency_raw")))
        )
    )
)

# Display original and cleaned values
nyc311_fields_df.select(
    "agency_raw",
    "agency"
).show(20, truncate=False)

# 31. Verify the cleaned agency column

In [ ]:
nyc311_fields_df.select(

    # Count missing cleaned agency values
    F.sum(
        F.when(
            F.col("agency").isNull(), 1
        ).otherwise(0)
    ).alias("missing_cleaned_agency"),

    # Count agency values changed during cleaning
    F.sum(
        F.when(
            F.col("agency_raw") != F.col("agency"), 1
        ).otherwise(0)
    ).alias("modified_agency"),

    # Count total requests
    F.count("*").alias("total_requests")

).show()

# 32. Create an overall Silver data-quality summary

In [ ]:
quality_summary_df = nyc311_fields_df.agg(

    F.count("*").alias("total_requests"),

    F.sum(F.when(F.col("bbl_status") == "missing", 1).otherwise(0))
     .alias("missing_bbl"),

    F.sum(F.when(F.col("bbl_status") == "invalid", 1).otherwise(0))
     .alias("invalid_bbl"),

    F.sum(F.when(F.col("borough_status") == "unexpected", 1).otherwise(0))
     .alias("unexpected_borough"),

    F.sum(F.when(F.col("community_board_status") == "special_area", 1).otherwise(0))
     .alias("special_area_board"),

    F.sum(F.when(F.col("community_board_status") == "unexpected", 1).otherwise(0))
     .alias("unexpected_board"),

    F.sum(F.when(F.col("incident_zip_status") == "missing", 1).otherwise(0))
     .alias("missing_zip"),

    F.sum(F.when(F.col("incident_zip_status") == "invalid_format", 1).otherwise(0))
     .alias("invalid_zip"),

    F.sum(F.when(F.col("complaint_type").isNull(), 1).otherwise(0))
     .alias("missing_complaint_type"),

    F.sum(F.when(F.col("complaint_detail").isNull(), 1).otherwise(0))
     .alias("missing_complaint_detail"),

    F.sum(F.when(F.col("agency").isNull(), 1).otherwise(0))
     .alias("missing_agency")
)

quality_summary_df.show(truncate=False, vertical=True)

In [ ]:
# One important finding is that not every NYC 311 request has a usable BBL.

# 33. Create a BBL join-eligibility flag

In [ ]:
# Create a flag showing whether the BBL can be used for joining

nyc311_fields_df = nyc311_fields_df.withColumn(
    "bbl_join_eligible",
    F.col("bbl_status") == "valid"
)

# Count eligible and ineligible requests
nyc311_fields_df.groupBy(
    "bbl_join_eligible"
).count().show()

# 34. Create an overall row-level data-quality flag

In [ ]:
# Add new column - has_invalid_field

nyc311_fields_df = nyc311_fields_df.withColumn(
    "has_invalid_field",

    (F.col("bbl_status") == "invalid") |
    (F.col("borough_status") == "unexpected") |
    (F.col("community_board_status") == "unexpected") |
    (F.col("incident_zip_status") == "invalid_format")
)

# Count requests with and without invalid fields
nyc311_fields_df.groupBy(
    "has_invalid_field"
).count().show()

# 35: FINAL NYC 311 SILVER FIELD TRANSFORMATIONS

In [ ]:
# ------------------------------------------------------------
# 1. Define valid reference values
# ------------------------------------------------------------

valid_boroughs = [
    "BRONX", "BROOKLYN", "MANHATTAN",
    "QUEENS", "STATEN ISLAND"
]

# Special Community Board codes retained from exploration
special_areas = [
    "64 MANHATTAN",
    "55 BROOKLYN", "56 BROOKLYN",
    "26 BRONX", "27 BRONX", "28 BRONX",
    "80 QUEENS", "81 QUEENS", "82 QUEENS",
    "83 QUEENS", "84 QUEENS",
    "95 STATEN ISLAND"
]

# Maximum numbered community districts by borough
max_board_numbers = {
    "BRONX": 12,
    "BROOKLYN": 18,
    "MANHATTAN": 12,
    "QUEENS": 14,
    "STATEN ISLAND": 3
}

# ------------------------------------------------------------
# 2. Reusable column expressions
# ------------------------------------------------------------

bbl_text = F.trim(F.col("BBL"))

board_text = F.upper(
    F.trim(F.col("Community Board"))
)

zip_text = F.trim(F.col("Incident Zip"))

detail_text = F.trim(
    F.col("Problem Detail (formerly Descriptor)")
)

# ------------------------------------------------------------
# 3. Rebuild the cleaned NYC 311 DataFrame
# ------------------------------------------------------------

nyc311_fields_df = (
    nyc311_raw_df

    # 3.1 Request identifier
    .withColumn(
        "request_id",
        F.trim(F.col("Unique Key"))
    )
    .withColumn(
        "request_id_status",
        F.when(
            F.col("request_id").isNull() |
            (F.col("request_id") == ""),
            "missing"
        ).when(
            F.col("request_id").rlike(r"^[0-9]+$"),
            "valid"
        ).otherwise("invalid")
    )

    # 3.2 BBL
    .withColumn("bbl_raw", F.col("BBL"))
    .withColumn(
        "bbl",
        F.when(
            bbl_text.rlike(r"^[1-5][0-9]{9}$"),
            bbl_text
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "bbl_status",
        F.when(
            F.col("bbl_raw").isNull() |
            (bbl_text == ""),
            "missing"
        ).when(
            F.col("bbl").isNotNull(),
            "valid"
        ).otherwise("invalid")
    )

    # 3.3 Borough
    .withColumn("borough_raw", F.col("Borough"))
    .withColumn(
        "borough",
        F.when(
            F.col("Borough").isNull() |
            (F.trim(F.col("Borough")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("Borough")))
        )
    )
    .withColumn(
        "borough_status",
        F.when(
            F.col("borough").isNull(), "missing"
        ).when(
            F.col("borough") == "UNSPECIFIED",
            "unspecified"
        ).when(
            F.col("borough").isin(valid_boroughs),
            "valid"
        ).otherwise("unexpected")
    )

    # 3.4 Community Board
    .withColumn(
        "community_board_raw",
        F.col("Community Board")
    )
    .withColumn(
        "community_board",
        F.when(
            board_text.rlike(
                r"^[0-9]{2} (BRONX|BROOKLYN|MANHATTAN|QUEENS|STATEN ISLAND)$"
            ),
            board_text
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "_board_number",
        F.substring("community_board", 1, 2).cast("int")
    )
    .withColumn(
        "_board_borough",
        F.regexp_extract(
            F.col("community_board"),
            r"^[0-9]{2} (.+)$",
            1
        )
    )
)

# Build maximum Community Board number expression
max_board_expr = F.lit(0)

for borough_name, maximum in max_board_numbers.items():
    max_board_expr = F.when(
        F.col("_board_borough") == borough_name,
        F.lit(maximum)
    ).otherwise(max_board_expr)

nyc311_fields_df = (
    nyc311_fields_df

    .withColumn(
        "_max_board_number",
        max_board_expr
    )
    .withColumn(
        "community_board_status",
        F.when(
            F.col("community_board_raw").isNull() |
            (board_text == ""),
            "missing"
        ).when(
            board_text.contains("UNSPECIFIED"),
            "unspecified"
        ).when(
            F.col("community_board").isin(special_areas),
            "special_area"
        ).when(
            F.col("community_board").isNotNull() &
            (F.col("_board_number") >= 1) &
            (F.col("_board_number") <= F.col("_max_board_number")),
            "valid_numbered"
        ).when(
            F.col("community_board").isNotNull(),
            "invalid_number"
        ).otherwise("unexpected")
    )

    # Remove helper columns
    .drop(
        "_board_number",
        "_board_borough",
        "_max_board_number"
    )

    # 3.5 Incident ZIP
    .withColumn(
        "incident_zip_raw",
        F.col("Incident Zip")
    )
    .withColumn(
        "incident_zip",
        F.when(
            zip_text.rlike(r"^[0-9]{5}$"),
            zip_text
        ).otherwise(F.lit(None).cast("string"))
    )
    .withColumn(
        "incident_zip_status",
        F.when(
            F.col("Incident Zip").isNull() |
            (zip_text == ""),
            "missing"
        ).when(
            F.col("incident_zip").isNotNull(),
            "valid_format"
        ).otherwise("invalid_format")
    )

    # 3.6 Complaint type
    .withColumn(
        "complaint_type_raw",
        F.col("Problem (formerly Complaint Type)")
    )
    .withColumn(
        "complaint_type",
        F.when(
            F.col("complaint_type_raw").isNull() |
            (F.trim(F.col("complaint_type_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("complaint_type_raw"))
        )
    )

    # 3.7 Complaint detail
    .withColumn(
        "complaint_detail_raw",
        F.col("Problem Detail (formerly Descriptor)")
    )
    .withColumn(
        "complaint_detail",
        F.when(
            F.col("complaint_detail_raw").isNull() |
            (detail_text == "") |
            (F.upper(detail_text) == "N/A"),
            F.lit(None).cast("string")
        ).otherwise(detail_text)
    )

    # 3.8 Agency
    .withColumn(
        "agency_raw",
        F.col("Agency")
    )
    .withColumn(
        "agency",
        F.when(
            F.col("agency_raw").isNull() |
            (F.trim(F.col("agency_raw")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(F.trim(F.col("agency_raw")))
        )
    )

    # 3.9 BBL join eligibility
    .withColumn(
        "bbl_join_eligible",
        F.col("bbl_status") == "valid"
    )

    # 3.10 Overall invalid field flag
    .withColumn(
        "has_invalid_field",
        (F.col("request_id_status") == "invalid") |
        (F.col("bbl_status") == "invalid") |
        (F.col("borough_status") == "unexpected") |
        (F.col("community_board_status").isin(
            "invalid_number", "unexpected"
        )) |
        (F.col("incident_zip_status") == "invalid_format")
    )
)

print("NYC 311 field transformations completed!")
print("Total columns:", len(nyc311_fields_df.columns))


# 36. Verify cleaned fields

In [ ]:
# 1. Check the DataFrame schema
print("=== CLEANED DATAFRAME SCHEMA ===")
nyc311_fields_df.printSchema()

# 2. Verify required output columns
required_columns = [
    "request_id",
    "request_id_status",
    "bbl",
    "bbl_status",
    "borough",
    "borough_status",
    "community_board",
    "community_board_status",
    "incident_zip",
    "incident_zip_status",
    "complaint_type",
    "complaint_detail",
    "agency",
    "bbl_join_eligible",
    "has_invalid_field"
]

missing_columns = [
    c for c in required_columns
    if c not in nyc311_fields_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

print("All required columns are present!")

# 3. Inspect a small number of cleaned requests
nyc311_fields_df.select(
    "request_id",
    "request_id_status",
    "bbl",
    "bbl_status",
    "borough",
    "community_board_status",
    "incident_zip_status",
    "complaint_type",
    "bbl_join_eligible",
    "has_invalid_field"
).show(10, truncate=False)


# 37. Final data-quality summary

In [ ]:
quality_summary_df = nyc311_fields_df.agg(

    F.count("*").alias("total_requests"),

    F.sum(F.when(
        F.col("request_id_status") == "missing", 1
    ).otherwise(0)).alias("missing_request_ids"),

    F.sum(F.when(
        F.col("request_id_status") == "invalid", 1
    ).otherwise(0)).alias("invalid_request_ids"),

    F.sum(F.when(
        F.col("bbl_status") == "missing", 1
    ).otherwise(0)).alias("missing_bbl"),

    F.sum(F.when(
        F.col("bbl_status") == "invalid", 1
    ).otherwise(0)).alias("invalid_bbl"),

    F.sum(F.when(
        F.col("borough_status") == "unexpected", 1
    ).otherwise(0)).alias("unexpected_borough"),

    F.sum(F.when(
        F.col("community_board_status") == "invalid_number", 1
    ).otherwise(0)).alias("invalid_board_numbers"),

    F.sum(F.when(
        F.col("community_board_status") == "special_area", 1
    ).otherwise(0)).alias("special_area_boards"),

    F.sum(F.when(
        F.col("incident_zip_status") == "invalid_format", 1
    ).otherwise(0)).alias("invalid_zip"),

    F.sum(F.when(
        F.col("complaint_type").isNull(), 1
    ).otherwise(0)).alias("missing_complaint_type"),

    F.sum(F.when(
        F.col("agency").isNull(), 1
    ).otherwise(0)).alias("missing_agency"),

    F.sum(F.when(
        F.col("bbl_join_eligible"), 1
    ).otherwise(0)).alias("bbl_join_eligible_count"),

    F.sum(F.when(
        F.col("has_invalid_field"), 1
    ).otherwise(0)).alias("rows_with_invalid_fields")
)

quality_summary_df.show(
    truncate=False,
    vertical=True
)

print("Field quality summary completed!")


# 38. Verify REQUEST ID Uniqueness

In [ ]:
duplicate_ids = (
    nyc311_fields_df
    .filter(F.col("request_id_status") == "valid")
    .groupBy("request_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Sample duplicate request IDs:")
duplicate_ids.show(10, truncate=False)

print("Number of duplicated request IDs:")
print(duplicate_ids.count())


# 39. Create validation flag and separate quarantine

In [ ]:
# Create validity flag and quarantine reason
nyc311_checked_df = (
    nyc311_fields_df
    .withColumn(
        "_silver_is_valid",
        F.col("request_id_status") == "valid"
    )
    .withColumn(
        "quality_reason",
        F.when(
            F.col("request_id_status") == "missing",
            "missing_request_id"
        ).when(
            F.col("request_id_status") == "invalid",
            "invalid_request_id"
        ).otherwise(F.lit(None).cast("string"))
    )
)

# Separate usable and unusable records
nyc311_valid_df = nyc311_checked_df.filter(
    F.col("_silver_is_valid") == True
)

nyc311_quarantine_df = nyc311_checked_df.filter(
    F.col("_silver_is_valid") == False
)

print("Valid-ID records:", nyc311_valid_df.count())
print("Quarantined records:", nyc311_quarantine_df.count())

print("Quarantine reasons:")
nyc311_quarantine_df.groupBy(
    "quality_reason"
).count().show()


# 40. Handle duplicate NYC 311 request IDs

In [ ]:
# Preserve all source columns for a repeatable selection
source_columns = sorted(nyc311_raw_df.columns)

ranked_df = nyc311_valid_df.withColumn(
    "_record_hash",
    F.sha2(
        F.to_json(
            F.struct(*[F.col(c) for c in source_columns])
        ),
        256
    )
)

# Select one source record per request ID
window_spec = (
    Window.partitionBy("request_id")
    .orderBy(F.col("_record_hash").asc())
)

ranked_df = ranked_df.withColumn(
    "_row_number",
    F.row_number().over(window_spec)
)

# Accepted records
nyc311_valid_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_record_hash", "_row_number")
)

# Additional records with the same request ID
duplicate_quarantine_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .withColumn("_silver_is_valid", F.lit(False))
    .withColumn(
        "quality_reason",
        F.lit("duplicate_request_id")
    )
    .drop("_record_hash", "_row_number")
)

# Combine both types of quarantined records
nyc311_quarantine_df = (
    nyc311_quarantine_df
    .unionByName(duplicate_quarantine_df)
)

print("Accepted requests:", nyc311_valid_df.count())
print("Quarantined requests:", nyc311_quarantine_df.count())


# 41. Verify final NYC 311 field quarantine

In [ ]:

original_count = nyc311_fields_df.count()
valid_count = nyc311_valid_df.count()
quarantine_count = nyc311_quarantine_df.count()

# Check no records were lost
assert original_count == valid_count + quarantine_count, (
    "ERROR: Row counts do not match!"
)

# Check accepted request IDs are unique
unique_count = (
    nyc311_valid_df
    .select("request_id")
    .distinct()
    .count()
)

assert valid_count == unique_count, (
    "ERROR: Duplicate request IDs remain!"
)

print("All quality checks passed!")

print("Original records:", original_count)
print("Valid records:", valid_count)
print("Quarantined records:", quarantine_count)

print("\nQuarantine breakdown:")
nyc311_quarantine_df.groupBy(
    "quality_reason"
).count().show()


# 42. Save valid and quarantined NYC 311 records as parquet

In [ ]:
BASE_OUTPUT = "/home/ubuntu/nyc311-data/dev"

valid_path = f"{BASE_OUTPUT}/silver/nyc311_fields_valid"
quarantine_path = f"{BASE_OUTPUT}/quarantine/nyc311_fields"

# Save accepted requests
nyc311_valid_df.write \
    .mode("overwrite") \
    .parquet(valid_path)

# Save quarantined requests
nyc311_quarantine_df.write \
    .mode("overwrite") \
    .parquet(quarantine_path)

print("Valid NYC 311 fields saved:", valid_path)
print("Quarantine saved:", quarantine_path)
